---

# Tugas Besar 2 IF3170: Prediksi Pembatalan Pemesanan Penginapan

*Eksperimen `nb_categorical_two_features`: Categorical Naive Bayes from scratch dengan dua fitur.*

---

## Kelompok 17 DerbyCounty

- Argha (penanggung jawab eksperimen Naive Bayes)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Preparation**](#3)
4. [**Modeling dan Validation**](#4)
5. [**Submission**](#5)

---

# Introduction <a name="1"></a>

---

## Aim

*Eksperimen ini mengevaluasi Categorical Naive Bayes dengan dua fitur dan membandingkan hasilnya dengan scikit-learn pada fold yang sama.*

## Approach

| Komponen | Pengaturan |
| --- | --- |
| Cleaning | Buang duplikat pada train dan tandai `num_adults = 0` sebagai nilai kosong |
| Fitur | Dua fitur |
| Transformasi | `NBPreprocessor` mengubah fitur menjadi kode kategori per kolom |
| Model | Categorical Naive Bayes dengan Laplace smoothing (`alpha=1`) |
| Validasi | Stratified 5-fold, seed 42, dan threshold dari OOF |

---

# Initialization <a name="2"></a>

---

## Import Libraries

In [1]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.naive_bayes import CategoricalNB

# Cari root repo dari direktori kerja notebook.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'models').is_dir())
sys.path.insert(0, str(ROOT / 'src'))

from models import NaiveBayes
from preprocessing import ID_COL, TARGET, clean_train
from preprocessing.nb import NBPreprocessor
from utils import (best_threshold, confusion_matrix, cross_val_proba, fold_scores,
                   load_model, save_model, stratified_kfold)

## Settings

Kedua implementasi memakai nilai `alpha` dan pilihan fitur yang sama. Alpha 1 menambahkan satu hitungan untuk setiap kategori di setiap kelas.

In [2]:
class Settings:
    SEED       = 42
    DATA_DIR   = ROOT / 'data'
    OUTPUT_DIR = ROOT / 'output'
    N_FOLDS    = 5

    FEATURES   = ('booking_channel', 'special_requests')
    ALPHA      = 1.0

    EXP_NAME   = 'nb_categorical_two_features'
    MODEL_PATH = OUTPUT_DIR / 'models' / f'{EXP_NAME}.pkl'
    SUB_PATH   = OUTPUT_DIR / 'submissions' / f'submission_{EXP_NAME}.csv'

CFG = Settings()
CFG.SUB_PATH.parent.mkdir(parents=True, exist_ok=True)

## Seed Everything

In [3]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

In [4]:
train_raw = pd.read_csv(CFG.DATA_DIR / 'train.csv', dtype={ID_COL: str})
test_raw  = pd.read_csv(CFG.DATA_DIR / 'test.csv', dtype={ID_COL: str})
sample    = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv', dtype={ID_COL: str})

print('train ', train_raw.shape)
print('test  ', test_raw.shape)

train  (27373, 19)
test   (6844, 18)


---

# Data Preparation <a name="3"></a>

---

## 3.1 Data Cleaning dan Transformation

Bersihkan duplikat pada data latih dengan fungsi bersama. Gunakan kanal pemesanan dan jumlah permintaan khusus sebagai kategori. Kamus kategori dibuat saat `fit` dan dipakai kembali saat `transform`. Kode 0 mewakili `Other`, sedangkan kode 1 mewakili nilai kosong.

In [5]:
train = clean_train(train_raw)
y = train[TARGET].to_numpy()


def make_prep():
    return NBPreprocessor(features=CFG.FEATURES)


X_check = make_prep().fit_transform(train)
print('sebelum', train_raw.shape, ' sesudah', train.shape)
print('fitur  ', X_check.shape[1])
print(list(X_check.columns))

sebelum (27373, 19)  sesudah (27309, 19)
fitur   2
['booking_channel', 'special_requests']


---

# Modeling dan Validation <a name="4"></a>

---

## 4.1 Validasi Silang

Pada setiap fold, preprocessor dan model dilatih dengan bagian train. Peluang dari bagian validasi digabung menjadi prediksi out-of-fold (OOF). Threshold yang memberi Macro F1 OOF tertinggi dipakai untuk prediksi akhir.

In [6]:
folds = stratified_kfold(y, n_splits=CFG.N_FOLDS, seed=CFG.SEED)

oof_nb = cross_val_proba(lambda: NaiveBayes(alpha=CFG.ALPHA), make_prep, train, y, folds)
THRESHOLD, oof_score = best_threshold(y, oof_nb)
per_fold = fold_scores(y, oof_nb, folds, THRESHOLD)

print(f'threshold      {THRESHOLD}')
print(f'Macro F1 OOF   {oof_score:.4f}')
print(f'per fold       {per_fold.round(4)}')
print(f'mean / std     {per_fold.mean():.4f} / {per_fold.std():.4f}')
print('confusion matrix [[TN, FP], [FN, TP]]')
print(confusion_matrix(y, (oof_nb >= THRESHOLD).astype(int)))

threshold      0.44
Macro F1 OOF   0.6652
per fold       [0.666  0.6596 0.6617 0.6711 0.6677]
mean / std     0.6652 / 0.0041
confusion matrix [[TN, FP], [FN, TP]]
[[15415  2950]
 [ 4671  4273]]


## 4.2 Perbandingan dengan scikit-learn

`CategoricalNB` menerima hasil preprocessing yang sama dan memakai minimal dua kode kategori per fitur agar cocok dengan kode cadangan pada implementasi from scratch.

In [7]:
# Ambil probabilitas kelas 1 agar keluaran sesuai dengan fungsi validasi tim.
class SklearnNB:
    def __init__(self):
        self.model = CategoricalNB(alpha=CFG.ALPHA, min_categories=2)

    def fit(self, X, y):
        self.model.fit(X, y)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)[:, 1]


oof_sk = cross_val_proba(SklearnNB, make_prep, train, y, folds)
thr_sk, score_sk = best_threshold(y, oof_sk)
per_fold_sk = fold_scores(y, oof_sk, folds, thr_sk)

comparison = pd.DataFrame({
    'OOF Macro F1': [oof_score, score_sk],
    'Std antar-fold': [per_fold.std(), per_fold_sk.std()],
    'Threshold': [THRESHOLD, thr_sk],
}, index=['From scratch', 'scikit-learn'])

print(f'Selisih maksimum peluang OOF: {np.abs(oof_nb - oof_sk).max():.2e}')
comparison.round(4)

Selisih maksimum peluang OOF: 6.11e-16


,OOF Macro F1,Std antar-fold,Threshold
From scratch,0.6652,0.0041,0.44
scikit-learn,0.6652,0.0041,0.44


#### Insights

> Skor dan threshold kedua implementasi sama pada konfigurasi ini. Selisih maksimum peluang OOF berada pada skala pembulatan bilangan pecahan.

---

# Submission <a name="5"></a>

---

## 5.1 Training Final, Save, dan Load

Model dilatih ulang dengan seluruh data train. Preprocessor, model, dan threshold disimpan dalam satu file, lalu dimuat kembali untuk prediksi.

In [8]:
prep_final = make_prep().fit(train)
nb_final = NaiveBayes(alpha=CFG.ALPHA).fit(prep_final.transform(train), y)
print('fitur', len(prep_final.columns_), 'threshold', THRESHOLD)

save_model({'preprocessor': prep_final, 'model': nb_final, 'threshold': THRESHOLD}, CFG.MODEL_PATH)
bundle = load_model(CFG.MODEL_PATH)

fitur 2 threshold 0.44


## 5.2 Prediksi dan File Submission

In [9]:
X_test = bundle['preprocessor'].transform(test_raw)
test_pred = bundle['model'].predict(X_test, threshold=bundle['threshold'])

submission = pd.DataFrame({ID_COL: test_raw[ID_COL], TARGET: test_pred})
assert submission[ID_COL].equals(sample[ID_COL])
submission.to_csv(CFG.SUB_PATH, index=False)

print(submission[TARGET].value_counts(normalize=True).round(4))
submission.head()

is_canceled
0    0.7439
1    0.2561
Name: proportion, dtype: float64


,id,is_canceled
0,5cb6f473,0
1,c06b4ab3,1
2,22bf3fc3,0
3,25408a48,1
4,621586b6,0


## 5.3 Contoh Satu Pemesanan

Pemesanan `e033a8b3` lewat Corporate dan memiliki satu permintaan khusus. Kamus di bawah menunjukkan kode yang dipakai model untuk menghitung peluang batal.

In [10]:
example = test_raw.loc[test_raw[ID_COL] == 'e033a8b3']
example_X = bundle['preprocessor'].transform(example)
probability = bundle['model'].predict_proba(example_X)[0]
prediction = bundle['model'].predict(example_X, threshold=THRESHOLD)[0]

print('kamus kanal', prep_final.category_codes_['booking_channel'])
print('kamus permintaan khusus', prep_final.category_codes_['special_requests'])
print(f'peluang batal {probability:.2%}, prediksi {prediction}')
example[[ID_COL, 'booking_channel', 'special_requests']]

kamus kanal {'Other': 0, '__MISSING__': 1, 'Airline': 2, 'Complimentary': 3, 'Corporate': 4, 'Offline Agent': 5, 'Online Travel Agent': 6}
kamus permintaan khusus {'Other': 0, '__MISSING__': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7}
peluang batal 7.71%, prediksi 0


,id,booking_channel,special_requests
284,e033a8b3,Corporate,1.0
